# AIS Spoofing Scenarios: Dutch North Sea Waters

This notebook demonstrates that **structurally valid and contextually plausible AIS messages** can be manually constructed using the validated `pyais` encoder and knowledge of real traffic distributions from the Exploratory Data Analysis (EDA).

Each scenario defines a threat narrative, constructs realistic AIS field values, encodes them into valid NMEA 0183 sentences, verifies the result through round-trip decoding, and provides a geographic visualization of where the spoofed vessel would appear.

All scenarios are situated in or near the **Netherlands' North Sea waters** and use:
- Valid MMSI prefixes for the claimed flag state
- Coordinates verified to be in water (not on land)
- Field values within ITU-R M.1371 valid ranges
- Values informed by EDA baseline distributions
- Structurally valid NMEA 0183 sentences produced by the `pyais` encoder (validated with 28/28 tests passing)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
from pyais import decode
from pyais.encode import encode_dict

print("Dependencies loaded successfully.")

Dependencies loaded successfully.


In [2]:
def encode_and_verify(vessel_data, label):
    """Encode a vessel dictionary, decode it back, print both, verify round-trip."""
    print(f"\n{'='*70}")
    print(f"  {label}")
    print(f"{'='*70}")

    print(f"\n  INPUT FIELD VALUES:")
    print(f"    MMSI:       {vessel_data['mmsi']}")
    print(f"    Lat:        {vessel_data['lat']}")
    print(f"    Lon:        {vessel_data['lon']}")
    print(f"    SOG:        {vessel_data['speed']} knots")
    print(f"    COG:        {vessel_data['course']} degrees")
    print(f"    Heading:    {vessel_data['heading']} degrees")
    print(f"    ROT:        {vessel_data.get('turn', 0.0)}")
    print(f"    Nav Status: {vessel_data['status']}")

    encoded = encode_dict(vessel_data)
    print(f"\n  ENCODED NMEA SENTENCE:")
    print(f"    {encoded[0]}")

    decoded = decode(*encoded).asdict()
    print(f"\n  DECODED BACK:")
    print(f"    MMSI:       {decoded['mmsi']}")
    print(f"    Lat:        {decoded['lat']}")
    print(f"    Lon:        {decoded['lon']}")
    print(f"    SOG:        {decoded['speed']} knots")
    print(f"    COG:        {decoded['course']} degrees")
    print(f"    Heading:    {decoded['heading']} degrees")
    print(f"    ROT:        {decoded['turn']}")
    print(f"    Nav Status: {decoded['status']}")

    # Verify round-trip
    checks = {
        'MMSI': str(decoded['mmsi']) == vessel_data['mmsi'],
        'SOG': decoded['speed'] == vessel_data['speed'],
        'COG': decoded['course'] == vessel_data['course'],
        'Heading': decoded['heading'] == vessel_data['heading'],
        'Lat': abs(decoded['lat'] - vessel_data['lat']) < 0.001,
        'Lon': abs(decoded['lon'] - vessel_data['lon']) < 0.001,
    }
    all_ok = all(checks.values())
    print(f"\n  ROUND-TRIP: {'PASS' if all_ok else 'FAIL'}")
    if not all_ok:
        for field, ok in checks.items():
            if not ok:
                print(f"    {field}: MISMATCH")

    return encoded, decoded

---
## Scenario 1: Ghost Vessel Blocking the Nieuwe Waterweg Entrance

### Threat Narrative

The Port of Rotterdam is Europe's largest seaport, handling approximately 30,000 sea-going vessel visits per year. All inbound and outbound deep-draught traffic must transit the **Nieuwe Waterweg**, the sole direct ship canal connecting Rotterdam to the North Sea. At the western end of this canal, near Hook of Holland, the **Maeslantkering** storm surge barrier marks the narrowest navigable section, a chokepoint just 480 to 675 metres wide, dredged to 14.5–16 metres depth.

In this scenario, an attacker injects a **single spoofed AIS position report** representing a large vessel broadcasting **navigational status 2 (not under command)** at **SOG 0.3 knots**, positioned diagonally across the Nieuwe Waterweg directly at the Maeslantkering barrier passage. The vessel's heading is set to 25° (NNE), oriented perpendicular to the channel that runs at approximately 110°/290° at this point, indicating the vessel has swung sideways across the fairway. This is the worst-case orientation for a drifting vessel in a narrow canal: it blocks the full navigable width.

### Why This Is Plausible

| Aspect | Justification |
|--------|---------------|
| **Location** | 51.955°N, 4.163°E —> Nieuwe Waterweg at the Maeslantkering passage, the narrowest point of Rotterdam's sea access |
| **MMSI** | 244810001 —> valid Dutch prefix (244), fictitious vessel number |
| **SOG = 0.0** | 61.3% of cleaned AIS records in our dataset have SOG = 0; stationary vessels are extremely common in AIS data |
| **Nav status = 2** | "Not under command" —> a valid ITU-R M.1371 navigational status code indicating loss of propulsion or steering |
| **Heading 160° ≠ COG 355°** | The vessel is pointing roughly south-southeast while drifting very slowly north —> consistent with a vessel that has lost steering and swung across the current |
| **ROT = 2.0** | Very slow rotation, consistent with passive drift in a tidal current |
| **Channel width** | At 480–675 m, a single large vessel oriented diagonally can obstruct the full navigable width |


### Potential Impact

- Vessel Traffic Services (VTS) must treat the AIS report as genuine, the protocol has no authentication mechanism
- All inbound **and** outbound traffic to the Port of Rotterdam would be halted until the obstruction is verified
- Tugs and coast guard vessels may be dispatched to assist a vessel that does not exist
- The Maeslantkering barrier operators may need to assess whether the "drifting" vessel poses a risk to the barrier structure itself
- Even minutes of disruption at this chokepoint cascades into supply chain delays for Europe's largest port

In [ ]:
# Scenario 1: Spoofed vessel field values
scenario_1 = {
    'type': 1,
    'mmsi': '244810001',       # Dutch MMSI prefix (244), fictitious vessel
    'status': 2,               # Not under command
    'turn': -1.5,              # Very slow counter-clockwise drift (tidal current)
    'speed': 0.3,              # Near-stationary, slight drift with current
    'accuracy': 1,             # High accuracy GPS
    'lon': 4.1628,             # Nieuwe Waterweg at Maeslantkering passage
    'lat': 51.9552,            # Narrowest point of Rotterdam sea access
    'course': 290.0,           # Drifting west-northwest with outgoing current
    'heading': 25,             # Vessel swung NNE, perpendicular to the channel
    'second': 15,
    'maneuver': 0,
    'raim': 0,
}

encoded_1, decoded_1 = encode_and_verify(
    scenario_1, "Scenario 1: Ghost vessel blocking Nieuwe Waterweg at Maeslantkering"
)


  Scenario 1: Ghost vessel blocking Nieuwe Waterweg at Maeslantkering

  INPUT FIELD VALUES:
    MMSI:       244810001
    Lat:        51.9552
    Lon:        4.1628
    SOG:        0.3 knots
    COG:        290.0 degrees
    Heading:    25 degrees
    ROT:        -1.5
    Nav Status: 2

  ENCODED NMEA SENTENCE:
    !AIVDO,1,1,,A,13aN14BvP3PC3TPMfb0;E0jN0000,0*6C

  DECODED BACK:
    MMSI:       244810001
    Lat:        51.9552
    Lon:        4.1628
    SOG:        0.3 knots
    COG:        290.0 degrees
    Heading:    25 degrees
    ROT:        -2.0
    Nav Status: 2

  ROUND-TRIP: PASS


### Geographic Context

![Scenario 1 — Overview](../data/processed/Scenario1_2026-04-02_180511.png)

![Scenario 1 — Zoomed](../data/processed/Scenario1_zoomed_2026-04-02_180511.png)

---
## Scenario 2: [To be designed]

*Placeholder for the second scenario — collision risk / autonomous vessel.*

---
## Scenario 3: [To be designed]

*Placeholder for the third scenario — deceptive presence / intelligence.*

---
## Bonus Scenario: [To be designed]

*Placeholder for the bonus AtoN scenario.*

---
## Summary

*To be completed after all scenarios are finalized.*